# Aariz HRNet-W32: bounded engineering pilot

29 landmarks · random initialization · Google AI Pro 200 CCUs/month · planned maximum 10 CCUs.

This notebook has **no full-training mode**. Do CPU preparation first. No test data is extracted. The 18 training-only exclusions leave 682/150 train/valid cases. Near-duplicate candidates remain unresolved; no patient-disjointness or clinical accuracy claim is permitted. ProMax 2D millimetre results are provisional. Dataset: [Aariz v1, CC BY 4.0](https://doi.org/10.6084/m9.figshare.27986417.v1); preserve the repository's license audit and attribution.

Use a pinned reviewed repository commit. This notebook never downloads pretrained weights. Budget enforcement uses a wall clock and manual billing evidence, not a Colab billing API. GPU setup, validation and persistence consume credits too.


In [ ]:
RUN_CPU_PREPARATION = False
RUN_PILOT = False
REPO_COMMIT = "PASTE_REVIEWED_40_CHARACTER_COMMIT_SHA"
CCU_START = None  # Actual account balance immediately before GPU activation
CCU_PER_HOUR = None  # Actual displayed rate for this session
GPU_ACTIVATED_AT = None  # ISO UTC timestamp, e.g. 2026-09-20T12:00:00+00:00
RUN_NAME = "pilot-001"  # New directory for every attempt; no automatic resume
if RUN_CPU_PREPARATION and RUN_PILOT:
    raise RuntimeError("Choose CPU preparation or GPU pilot, never both")

## 1. Mount storage and pin source (CPU runtime)

In Colab choose **Runtime → Change runtime type → CPU** for preparation. Mount your own Drive. Do not put clinical patient images in this research workspace. If the repository is private, upload/check out the reviewed source with your usual credentials; do not paste tokens into notebook cells or saved output. This cell requires Python 3.12, the tested research runtime.


In [ ]:
import os
import re
import sys
import subprocess
from pathlib import Path
from google.colab import drive

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        "Use Python 3.12; revalidate dependency pins before changing versions"
    )
if not re.fullmatch(r"[0-9a-f]{40}", REPO_COMMIT):
    raise RuntimeError("Set REPO_COMMIT to the reviewed PR commit first")
drive.mount("/content/drive")
STORE = Path("/content/drive/MyDrive/DentalFlow-Aariz")
STORE.mkdir(parents=True, exist_ok=True)
REPO = Path("/content/cephalometric-ai")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/akhastaf/cephalometric-ai.git", str(REPO)],
        check=True,
    )
subprocess.run(["git", "fetch", "origin", REPO_COMMIT], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", REPO_COMMIT], cwd=REPO, check=True)
if subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=REPO, text=True
).strip():
    raise RuntimeError("Research checkout must be clean")
os.chdir(REPO)

## 2. Install pinned research dependencies

CPU preparation uses CPU PyTorch wheels. After switching to GPU, repeat source setup and install CUDA wheels. Installation on GPU counts against the session budget: record GPU activation time before doing it. If installation exhausts the allowance, disconnect rather than extending the pilot. Source and model weights have separate licensing; no weights are loaded here.


In [ ]:
if RUN_CPU_PREPARATION:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "torch==2.8.0",
            "--index-url",
            "https://download.pytorch.org/whl/cpu",
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "numpy==2.2.6",
            "Pillow==11.3.0",
            "pytest==9.0.2",
        ],
        check=True,
    )
elif RUN_PILOT:
    if any(v is None for v in (CCU_START, CCU_PER_HOUR, GPU_ACTIVATED_AT)):
        raise RuntimeError(
            "Record actual billing balance/rate and GPU activation time before setup"
        )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "torch==2.8.0",
            "--index-url",
            "https://download.pytorch.org/whl/cu128",
        ],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "numpy==2.2.6", "Pillow==11.3.0"],
        check=True,
    )
else:
    print("Preparation and pilot are disabled")

## 3. Download and verify the exact release; materialize train/valid only

Run on CPU. The archive is about 2.1 GB; keep sufficient space for extracted train/valid files. The checksum is pinned in `data.py`, and every extracted image/annotation is checked against the audit manifest. No test members are extracted or parsed. Failed/partial preparation must be removed manually before retrying into a fresh directory. Keep the original immutable archive.


In [ ]:
if RUN_CPU_PREPARATION:
    from research.aariz.pilot.data import prepare, digest, ARCHIVE_SHA256

    archive = STORE / "Aariz.zip"
    if not archive.exists():
        partial = STORE / "Aariz.zip.part"
        subprocess.run(
            [
                "curl",
                "--fail",
                "--location",
                "--retry",
                "3",
                "--max-time",
                "3600",
                "--output",
                str(partial),
                "https://ndownloader.figshare.com/files/51041642",
            ],
            check=True,
        )
        if digest(partial) != ARCHIVE_SHA256:
            raise RuntimeError("Archive checksum failed; do not extract")
        partial.rename(archive)
    prepared = STORE / "prepared-v1"
    if not prepared.exists():
        print(prepare(archive, prepared))
    else:
        print("Existing preparation will be fully verified by the dataset constructors")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "-q",
            "research/aariz/test_review_package.py",
            "research/aariz/pilot/test_pilot.py",
        ],
        check=True,
    )

## 4. Switch to GPU and capture billing evidence

Finish CPU preparation first. Select one available GPU, record the activation timestamp, actual CCUs remaining, displayed CCU/hour, and an evidence reference from the Colab resource panel. Close other paid runtimes. Re-run cells 0–2 with **RUN_CPU_PREPARATION=False** and **RUN_PILOT=True**. Use a new RUN_NAME each time. Changes in rate require stopping and reviewing the budget.

The process uses the earliest of 3 epochs, 250 optimizer-step attempts and the remaining wall allowance, with 20% credit headroom and a 60-second persistence reserve. Up to five minutes is reserved for validation. It reports incomplete validation explicitly. A short random-initialized pilot tests cost and mechanics, not clinical performance.


In [ ]:
if RUN_PILOT:
    import json
    import shutil
    from datetime import datetime, timezone
    from research.aariz.pilot.run import budget_seconds

    activated = datetime.fromisoformat(GPU_ACTIVATED_AT.replace("Z", "+00:00"))
    if activated.tzinfo is None:
        raise RuntimeError("Activation time needs a timezone")
    remaining = (
        budget_seconds(float(CCU_PER_HOUR), float(CCU_START))
        - (datetime.now(timezone.utc) - activated).total_seconds()
    )
    if remaining <= 60:
        raise RuntimeError("No pilot budget left after setup; disconnect GPU")
    try:
        LOCAL_DATA = Path("/content/aariz-pilot-data")
        if not LOCAL_DATA.exists():
            shutil.copytree(STORE / "prepared-v1", LOCAL_DATA)
        output = STORE / "runs" / RUN_NAME
        # The CLI checks the elapsed session time again after data staging.
        environment = dict(
            os.environ, CUBLAS_WORKSPACE_CONFIG=":4096:8", OMP_NUM_THREADS="2"
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "research.aariz.pilot.run",
                "--data",
                str(LOCAL_DATA),
                "--output",
                str(output),
                "--authorize-pilot",
                "--ccu-start",
                str(CCU_START),
                "--ccu-per-hour",
                str(CCU_PER_HOUR),
                "--gpu-activated-at",
                GPU_ACTIVATED_AT,
            ],
            check=True,
            env=environment,
            timeout=max(
                1,
                budget_seconds(float(CCU_PER_HOUR), float(CCU_START))
                - (datetime.now(timezone.utc) - activated).total_seconds(),
            ),
        )
    finally:
        # Save remaining outputs to Drive, then release the paid runtime.
        from google.colab import runtime

        runtime.unassign()

## 5. After disconnect: settle the actual CCU ledger on CPU

Switch to CPU, mount Drive, reopen the report. Wait for billing to settle and enter the ending balance. Record any grants/top-ups, concurrent sessions or billing uncertainty. Never label rate × time as actual consumption. Share report.json, validation.json (if generated), pip-freeze.txt and billing evidence for review. No full experiment starts automatically. Do not upload checkpoints or data to Git.


In [ ]:
CCU_END = None
CCU_GRANTS = 0
CONCURRENT_OR_UNCERTAIN_BILLING = True
BILLING_EVIDENCE_REFERENCE = ""

if CCU_END is not None:
    import json
    import math

    report_path = STORE / "runs" / RUN_NAME / "report.json"
    report = json.loads(report_path.read_text())
    values = (float(report["ccu_start"]), float(CCU_END), float(CCU_GRANTS))
    if not all(math.isfinite(v) and v >= 0 for v in values):
        raise ValueError("Invalid billing balances")
    consumed = values[0] + values[2] - values[1]
    measured = (
        not CONCURRENT_OR_UNCERTAIN_BILLING
        and bool(BILLING_EVIDENCE_REFERENCE)
        and consumed >= 0
    )
    report.update(
        ccu_end=CCU_END,
        ccu_grants=CCU_GRANTS,
        billing_evidence_reference=BILLING_EVIDENCE_REFERENCE,
        actual_ccu_consumed=consumed if measured else None,
        billing_status="MEASURED" if measured else "INCONCLUSIVE",
    )
    report_path.write_text(json.dumps(report, indent=2, allow_nan=False))
    print("Ledger saved. Full training still requires review.")

## Review before any full experiment

Review measured credit consumption, time/step, peak VRAM, finite gradients, all 29 outputs, image/point alignment, validation completeness and failures. Resolve or explicitly qualify remaining near-duplicate and device calibration issues. Freeze any full-run architecture, decoder, confidence calibration and selection policy before final test access. Export, ONNX parity, CPU latency/RAM, clinical acceptance and DentalFlow integration are later stages.
